# ShopTalk-X — Exploratory Data Analysis

Dataset: Amazon Berkeley Objects (ABO), English-language subset, produced by
`src/shoptalk/data/download_abo.py` + `src/shoptalk/data/preprocess.py`.

This notebook covers (per the grading rubric's EDA section):
- Category / brand / price distributions
- Description length & vocabulary analysis
- Image availability per product
- Missing-value analysis
- NLP preprocessing rationale (tokenization/normalization choices)
- Insights explicitly tied to modeling decisions (e.g. category imbalance -> hard-negative sampling)

**Note on price:** ABO does not publish price data. `price_usd` here is a
**synthetic**, category-band-seeded value (see `preprocess.py`) added only so
price-filtering queries ("under $50") are demoable. It is not real Amazon
pricing and should never be reported as such.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_colwidth", 100)

df = pd.read_parquet("../data/processed/products.parquet")
print(f"{len(df)} products, {df.shape[1]} columns")
df.head(3)

## 1. Missing-value analysis

In [ ]:
missing = df.isnull().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(1)
pd.DataFrame({"missing_count": missing, "missing_pct": missing_pct})

In [ ]:
empty_str_cols = ["item_name", "category", "brand", "color", "style", "description", "keywords"]
empty_counts = (df[empty_str_cols] == "").sum()
print("Empty-string (field present but blank) counts:")
empty_counts

**Insight:** `item_name`/`description` are near-complete (English filter in
preprocessing guarantees at least one is non-empty). `color`, `style`, and
`keywords` are the fields most often blank — these are optional ABO attributes
that vary heavily by category (e.g. furniture rarely tags "style" the way
apparel does). This motivates concatenating *all* available text fields into
one `document` field for embedding rather than relying on any single field.

## 2. Category distribution

In [ ]:
top_n = 25
cat_counts = df["category"].value_counts().head(top_n)

plt.figure(figsize=(10, 8))
sns.barplot(x=cat_counts.values, y=cat_counts.index, orient="h")
plt.xlabel("Product count")
plt.ylabel("Category (product_type)")
plt.title(f"Top {top_n} product categories")
plt.tight_layout()
plt.show()

print(f"{df['category'].nunique()} unique categories total")
print(f"Top 5 categories cover {cat_counts.head(5).sum() / len(df) * 100:.1f}% of products")

**Modeling decision:** category counts are heavily right-skewed (a handful of
categories dominate the catalog). This class imbalance means naive random
negative sampling for embedding fine-tuning would mostly draw *easy*
negatives from the long tail. We instead mine **hard negatives from
within the same category** (nearest neighbors in the ANN index, same
`product_type`, different `item_id`) so the model learns fine-grained
distinctions among visually/textually similar products — see
`ShopTalk-X_Production_Design_Document.md` §3.1.

## 3. Brand distribution

In [ ]:
top_brands = df["brand"].replace("", pd.NA).value_counts().head(20)

plt.figure(figsize=(10, 6))
sns.barplot(x=top_brands.values, y=top_brands.index, orient="h")
plt.xlabel("Product count")
plt.ylabel("Brand")
plt.title("Top 20 brands by product count")
plt.tight_layout()
plt.show()

print(f"{df['brand'].replace('', pd.NA).nunique()} unique brands")

## 4. Price distribution (synthetic — see note above)

In [ ]:
assert df["price_is_synthetic"].all(), "expected all prices to be flagged synthetic"

plt.figure(figsize=(9, 5))
sns.histplot(df["price_usd"], bins=40, kde=True)
plt.xlabel("Synthetic price (USD)")
plt.title("Synthetic price distribution (NOT real Amazon pricing)")
plt.tight_layout()
plt.show()

df["price_usd"].describe()

## 5. Description length & vocabulary analysis

In [ ]:
df["description_word_count"] = df["description"].str.split().str.len().fillna(0)
df["document_word_count"] = df["document"].str.split().str.len().fillna(0)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
sns.histplot(df["description_word_count"], bins=40, ax=axes[0])
axes[0].set_title("Description length (words)")
axes[0].set_xlabel("word count")

sns.histplot(df["document_word_count"], bins=40, ax=axes[1], color="darkorange")
axes[1].set_title("Full document length (words) — what gets embedded")
axes[1].set_xlabel("word count")
plt.tight_layout()
plt.show()

df[["description_word_count", "document_word_count"]].describe()

In [ ]:
from collections import Counter
import re

STOPWORDS = {
    "the", "a", "an", "and", "or", "for", "with", "of", "to", "in", "on",
    "is", "are", "this", "that", "your", "you", "it", "its", "by", "as",
    "from", "at", "be", "will", "not", "no",
}

def tokenize(text):
    return [t for t in re.findall(r"[a-z']+", text.lower()) if t not in STOPWORDS and len(t) > 2]

all_tokens = df["document"].map(tokenize).sum()
vocab = Counter(all_tokens)
print(f"vocabulary size (post-stopword filter): {len(vocab)}")
print("most common tokens:")
vocab.most_common(25)

**NLP preprocessing rationale:**
- **No stemming/lemmatization** applied before embedding — modern sentence
  embedding models (bge/e5) are trained on raw, unstemmed text and perform
  worse on artificially normalized input; stemming is reserved for the
  lexical/BM25 side of a future hybrid-search extension.
- **No lowercasing before embedding** — bi-encoders are case-aware and brand
  names / model numbers (e.g. "iPhone", "RGB") carry signal; lowercasing is
  only applied here for *vocabulary analysis*, not for the embedded text.
- **Whitespace/zero-width-space normalization** — ABO source text contains
  stray unicode whitespace artifacts (see `clean_text()` in `preprocess.py`)
  that add noise to tokenization without normalization.
- **Field concatenation with `" | "` delimiters** — keeps a lightweight
  structural signal (name vs. category vs. description) inside a single
  string, which cross-encoders in particular can exploit at rerank time.

## 6. Image availability

In [ ]:
avail = df["image_available"].value_counts(normalize=True) * 100
print(avail)

plt.figure(figsize=(4, 4))
df["image_available"].value_counts().plot.pie(autopct="%.1f%%", labels=["available", "missing"] if df["image_available"].sum() < len(df) else ["available"])
plt.ylabel("")
plt.title("Product image availability")
plt.show()

**Insight:** products missing a resolvable image (broken `main_image_id` /
absent from `images.csv`) cannot participate in CLIP image-search or
verification-head training — they are still kept for text-only retrieval but
should be excluded from the Day-3 CLIP indexing step and the Day-5
verification-pair construction.

## 7. Caption-vs-description overlap

In [ ]:
# Placeholder: BLIP captions are generated in the Day-3 captioning batch job
# (src/shoptalk/data/caption_images.py, not yet run). Once `caption` is added
# to products.parquet, re-run this cell to compare vocabulary overlap between
# generated captions and existing text descriptions -- the design doc's
# hypothesis is that captions surface *visual* attributes (pattern, shape,
# material appearance) that sellers often omit from bullet points.
if "caption" in df.columns:
    caption_tokens = set(df["caption"].map(tokenize).sum())
    desc_tokens = set(df["description"].map(tokenize).sum())
    overlap = len(caption_tokens & desc_tokens) / max(len(caption_tokens), 1) * 100
    print(f"caption vocab: {len(caption_tokens)}, description vocab: {len(desc_tokens)}")
    print(f"{overlap:.1f}% of caption vocabulary already appears in descriptions")
else:
    print("captions not yet generated -- run Day 3's captioning job, then re-run this cell")

## 8. Summary of modeling decisions driven by this EDA

| Observation | Decision |
|---|---|
| Heavy category imbalance (top 5 categories >> long tail) | Hard-negative mining scoped **within category**, not globally random |
| `color`/`style`/`keywords` frequently blank | Embed a concatenated `document` field, not any single attribute |
| Document length varies widely (some near-empty, some very long) | Truncate/pool embeddings at the model's max sequence length; monitor truncation rate as a data-quality metric |
| No native price field | Synthetic, clearly-flagged price used only for demo filtering; excluded from any trained model's ground truth |
| Some products have no resolvable image | Excluded from CLIP/verification pipelines (Day 3, 5); retained for text-only retrieval |
| Stray unicode whitespace in source text | Explicit cleaning step in `preprocess.py`, applied before embedding |